In [ ]:
from discovery_utils.utils.llm import batch_check

from functools import reduce
import os
import pandas as pd
import time

from dsp_interview_transcripts import config, PROJECT_DIR, logging
from dsp_interview_transcripts.getters.raw import get_raw_transcripts_cleaned

In [ ]:
def format_row(row):
    """Paste specific cols from a df into a text format that we can feed to GPT"""
    return f"{row['uuid']} | {row['role']} | {row['text']}"

def convert_transcripts_df_to_dict(data):
    """Returns a dict where the keys are conversation IDs and the values are the full text of the conversation."""
    # Convert each interview into one string
    conversation_texts = data.groupby('conversation').apply(lambda group: "\n".join(group.apply(format_row, axis=1)))

    return conversation_texts.to_dict()

In [ ]:
data = get_raw_transcripts_cleaned()

In [ ]:
# NB this is not the full / final list of questions
question_dict = {"found_house_cold": "Did the user find the house cold as a result of the trial?",
                 "noticed_temp_control": "Did the user notice that their temperature was being controlled?",
                 "desire_more_info": "Did the user seem to desire more information about the trial?",
                 "other_household_members": "Did other household members have a differing experience of the trial than the user?",}

In [ ]:
system_template = """
I will show you an interview between a person and a bot interviewer. The person has taken part in a trial
that allows their heating provider to control their heating remotely.

Answer the following question: {question}

If applicable, provide the text from the interview that led you to your answer, and its identifier in the transcript.

The interview will be given to you in the format:
```
<identifier-1> | BOT | <text>
<identifier-2>  | USER | <text>
<identifier-3>  | BOT | <text>
```
"""

In [ ]:
# Create a modified prompt and set of output fields for each question
question_prompt_dict = {}

for k, v in question_dict.items():
    question_prompt_dict[k] = {}
    system_message = system_template.format(question=v)
    fields = [
    {"name": k, "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    {"name": "explanation", "type": "str", "description": "Explain why you answered in the way you did."},
    {"name": "text", "type": "str", "description": "The text in the transcript where you found the answer."},
    {"name": "identifier", "type": "str", "description": "The identifier in the transcript where you found the answer."},
    ]
    
    question_prompt_dict[k]["system_message"] = system_message
    question_prompt_dict[k]["fields"] = fields

In [ ]:
conversation_text_dict = convert_transcripts_df_to_dict(data)

In [ ]:
# Just test on the one conversation for now
test_conv = '0194dc56-8600-72b6-c75d-52bc3553d98d'

test_data = {test_conv: conversation_text_dict[test_conv]}

In [ ]:
# Run the process for each question x every conversation in the data (just 1 in test_data)
for q in question_prompt_dict:

    processor = batch_check.LLMProcessor(
        model_name="gpt-4o-mini",
        temperature=0,
        output_path=f"{q}_output.jsonl",
        system_message=question_prompt_dict[q]["system_message"],
        session_name=q,
        output_fields=question_prompt_dict[q]["fields"],
    )

    processor.run(test_data, batch_size=1, sleep_time=0.5)

In [ ]:
time.sleep(120) # Pause here to wait until the outputs are ready

In [ ]:
# Read the output back in and merge the output for each question on conversation id

jsonl_files = [f"{q}_output.jsonl" for q in question_dict.keys()]
dfs = [pd.read_json(file, lines=True) for file in jsonl_files]

output_dfs = {}

cols_to_rename = ['explanation', 'text', 'identifier',
       'timestamp', 'model', 'temperature']

for q in question_prompt_dict:
    df = pd.read_json(f"{q}_output.jsonl", lines=True)
    df = df.rename(columns={col: f"{q}_{col}" for col in cols_to_rename})
    output_dfs[q] = df
    
dfs = list(output_dfs.values())

merged_df = reduce(lambda left, right: pd.merge(left, right, on="id", how="outer"), dfs)
merged_df.head()

In [ ]:
# Add in some basic checks to make sure nothing is hallucinated
def check_for_hallucinated_ids(output_data, original_data):
    identifier_cols = [col for col in output_data.columns if col.endswith('_identifier')]
    unique_identifiers = output_data[identifier_cols].stack().dropna().unique()

    hallucinated_ids = []

    for val in unique_identifiers:
        if val not in original_data['uuid'].values:
            hallucinated_ids.append(val)
    
    if len(hallucinated_ids) > 0:
        print(f"Hallucinated IDs found: {hallucinated_ids}")
        return hallucinated_ids
    else:
        print("No hallucinated IDs found.")
        return None
    
def check_for_hallucinated_text(output_data, original_data):
    text_cols = [col for col in output_data.columns if col.endswith('_text')]
    returned_text = output_data[text_cols].stack().dropna().unique()

    all_text = original_data['text'].unique()

    hallucinated_text = []

    for text in returned_text:
        if not any(text in full_text for full_text in all_text):  # Check for substring match
            hallucinated_text.append(text)
            
    if len(hallucinated_text) > 0:
        print(f"Hallucinated text found: {hallucinated_text}")
        return hallucinated_text
    else:
        print("No hallucinated text found.")
        return None

In [ ]:
check_for_hallucinated_ids(merged_df, data)

In [ ]:
check_for_hallucinated_text(merged_df, data)